# **ArguAna RLC: True Zero-Shot Offline Reinforcement Learning**
This notebook addresses the methodological updates:
1. **Offline Training on MS MARCO**: The SAE and PPO agents are trained *only* on MS MARCO.
2. **Observable State & Reward**: The agent uses Retrieval Entropy and Facet Variance (no target rank leakage). The reward uses an IPS click-model formulation.
3. **Advanced Baselines**: Rocchio (VPRF), HyDE (Hypothetical Document Embeddings), SPLADE, ColBERT.
4. **SAE Fidelity**: Spearman correlation check on MIPS to verify L1 topological preservation.


In [ ]:
# ==============================================================================
# STEP 0: Dependencies & Setup
# ==============================================================================
import os
os.system("pip install -q beir sentence-transformers transformers torch numpy scipy tqdm seaborn matplotlib huggingface_hub accelerate")

import gc
import warnings
import math
import logging
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F_func
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from beir import util
from beir.datasets.data_loader import GenericDataLoader

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n--- System Utilizing: {DEVICE.type.upper()} ---")

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_ndcg_at_k(retrieved_indices, relevant_indices, k=10):
    dcg = 0.0
    for i, idx in enumerate(retrieved_indices[:k]):
        if idx in relevant_indices:
            dcg += 1.0 / math.log2(i + 2)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_indices), k)))
    return dcg / idcg if idcg > 0 else 0.0


In [ ]:
# ==============================================================================
# STEP 1: LOAD MS MARCO (TRAINING) AND ARGUANA (EVALUATION)
# ==============================================================================
def load_dataset(dataset_name, sample_q_size=800, sample_d_size=15000):
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
    out_dir = util.download_and_unzip(url, "datasets")
    split = "dev" if dataset_name == "msmarco" else "test"
    corpus, queries, qrels = GenericDataLoader(data_folder=out_dir).load(split=split)
    
    query_ids = []
    relevant_doc_ids = []
    for qid in list(queries.keys()):
        if qid in qrels:
            valid_targets = [did for did, score in qrels[qid].items() if score > 0 and did in corpus]
            if valid_targets:
                query_ids.append(qid)
                relevant_doc_ids.extend(valid_targets)
        if len(query_ids) >= sample_q_size: break
    
    needed = max(0, sample_d_size - len(relevant_doc_ids))
    all_doc_ids = sorted(list(set(relevant_doc_ids + list(corpus.keys())[:needed])))
    
    doc_id_to_idx = {did: i for i, did in enumerate(all_doc_ids)}
    docs = [corpus[did].get("text") for did in all_doc_ids]
    qs = [queries[qid] for qid in query_ids]
    target_idx = [[doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx] for qid in query_ids]
    
    print(f"-> LOADED {dataset_name.upper()}: {len(qs)} Queries, {len(docs)} Documents")
    return qs, docs, query_ids, qrels, doc_id_to_idx, target_idx

print("Loading Disjoint Training Set (MS MARCO)...")
ms_q, ms_d, _, _, _, ms_targets = load_dataset("msmarco", sample_q_size=5000, sample_d_size=20000)

print("\nLoading Zero-Shot Evaluation Set (ArguAna)...")
arg_q, arg_d, arg_qids, arg_qrels, arg_d2idx, arg_targets = load_dataset("arguana", sample_q_size=800, sample_d_size=8500)


In [ ]:
# ==============================================================================
# STEP 2: BASELINES (ROCCHIO, HYDE, CONTRIEVER) ON ARGUANA
# ==============================================================================
contriever = SentenceTransformer('facebook/contriever', device=DEVICE)

print("Encoding ArguAna for Evaluation...")
with torch.no_grad():
    arg_q_dense = F_func.normalize(torch.tensor(contriever.encode(arg_q, batch_size=128, show_progress_bar=False)), p=2, dim=1).to(DEVICE)
    arg_d_dense = F_func.normalize(torch.tensor(contriever.encode(arg_d, batch_size=128, show_progress_bar=True)), p=2, dim=1).to(DEVICE)

# 1. Base Contriever
base_scores = torch.matmul(arg_q_dense, arg_d_dense.T)
base_ndcgs = [compute_ndcg_at_k(torch.topk(base_scores[i], 10).indices.tolist(), arg_targets[i], 10) for i in range(len(arg_q))]
print(f"\n[Baseline] Base Contriever NDCG@10: {np.mean(base_ndcgs):.4f}")

# 2. Vector PRF (Rocchio)
def evaluate_rocchio(q_dense, d_dense, targets, alpha=1.0, beta=0.8, top_prf=3):
    scores = torch.matmul(q_dense, d_dense.T)
    top_docs = d_dense[torch.topk(scores, top_prf, dim=1).indices]
    prf_mean = top_docs.mean(dim=1)
    q_rocchio = F_func.normalize(alpha * q_dense + beta * prf_mean, p=2, dim=1)
    
    new_scores = torch.matmul(q_rocchio, d_dense.T)
    ndcgs = [compute_ndcg_at_k(torch.topk(new_scores[i], 10).indices.tolist(), targets[i], 10) for i in range(len(q_dense))]
    return np.mean(ndcgs)

rocchio_ndcg = evaluate_rocchio(arg_q_dense, arg_d_dense, arg_targets)
print(f"[Baseline] Rocchio Vector PRF NDCG@10: {rocchio_ndcg:.4f}")

# 3. HyDE (Hypothetical Document Embeddings via flan-t5)
def evaluate_hyde(queries, query_dense, d_dense, targets):
    print("Generating HyDE Docs (Zero-Shot)...")
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
    gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(DEVICE)
    
    hyde_queries = []
    for i in tqdm(range(0, len(queries), 32)):
        batch = queries[i:i+32]
        prompts = [f"Please write a document that answers the following query: {q}" for q in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        outputs = gen_model.generate(**inputs, max_new_tokens=50)
        hyde_queries.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    
    del gen_model, tokenizer; clear_vram()
    
    hyde_q_dense = F_func.normalize(torch.tensor(contriever.encode(hyde_queries, batch_size=128, show_progress_bar=False)), p=2, dim=1).to(DEVICE)
    # Combine original query with hypothetical doc embedding (standard HyDE practice)
    combined_q = F_func.normalize(query_dense + hyde_q_dense, p=2, dim=1)
    
    scores = torch.matmul(combined_q, d_dense.T)
    ndcgs = [compute_ndcg_at_k(torch.topk(scores[i], 10).indices.tolist(), targets[i], 10) for i in range(len(queries))]
    return np.mean(ndcgs)

hyde_ndcg = evaluate_hyde(arg_q, arg_q_dense, arg_d_dense, arg_targets)
print(f"[Baseline] HyDE (flan-t5-small) NDCG@10: {hyde_ndcg:.4f}")

# ConSharp / IMRNNs:
print("[Baseline] ConSharp / IMRNNs require extensive offline pre-training on MS MARCO. For fair comparison, we refer to their cited metrics in the paper.")


In [ ]:
# ==============================================================================
# STEP 3: OFFLINE TRAINING ON MS MARCO (Zero-Shot Setup)
# ==============================================================================
class SparseStateEncoder(nn.Module):
    def __init__(self, dense_dim=768, sparse_dim=4096):
        super().__init__()
        self.encoder = nn.Linear(dense_dim, sparse_dim)
        self.relu = nn.ReLU()
        self.decoder = nn.Linear(sparse_dim, dense_dim)
    def forward(self, x):
        F = self.relu(self.encoder(x))
        return F, self.decoder(F)

print("\nEncoding MS MARCO for Offline Training...")
with torch.no_grad():
    ms_q_dense = F_func.normalize(torch.tensor(contriever.encode(ms_q, batch_size=256)), p=2, dim=1).to(DEVICE)
    ms_d_dense = F_func.normalize(torch.tensor(contriever.encode(ms_d, batch_size=256)), p=2, dim=1).to(DEVICE)

print("Training SAE strictly on MS MARCO...")
sae_layer = SparseStateEncoder().to(DEVICE)
opt = optim.Adam(sae_layer.parameters(), lr=1e-3)
mse = nn.MSELoss()
all_ms = torch.cat([ms_q_dense, ms_d_dense])

for epoch in tqdm(range(20), desc="SAE Epochs"):
    idx = torch.randperm(all_ms.size(0))
    for i in range(0, all_ms.size(0), 1024):
        batch = all_ms[idx[i:i+1024]]
        opt.zero_grad()
        F, rec = sae_layer(batch)
        loss = mse(rec, batch) + (1e-4 * F.mean())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sae_layer.parameters(), 1.0)
        opt.step()

with torch.no_grad():
    ms_q_F = F_func.normalize(sae_layer(ms_q_dense)[0], p=2, dim=1)
    ms_d_F = F_func.normalize(sae_layer(ms_d_dense)[0], p=2, dim=1)
    dec_weight = sae_layer.decoder.weight.detach()

# SAE FIDELITY PROOF
print("\nComputing Topographical Fidelity (Spearman Rank Correlation)...")
q_idx = 0
orig_scores = torch.matmul(ms_q_dense[q_idx], ms_d_dense.T).cpu().numpy()
rec_q = sae_layer(ms_q_dense[q_idx].unsqueeze(0))[1]
rec_d = sae_layer(ms_d_dense)[1]
rec_scores = torch.matmul(F_func.normalize(rec_q, p=2, dim=1), F_func.normalize(rec_d, p=2, dim=1).T).squeeze(0).detach().cpu().numpy()

correlation, _ = stats.spearmanr(orig_scores, rec_scores)
print(f"Topological Preservation Correlation: {correlation:.4f} (Ideal > 0.95)")


In [ ]:
# ==============================================================================
# STEP 4: LABEL-LEAKAGE-FREE REINFORCEMENT LEARNING
# ==============================================================================
class LeakFreeSteeringEnv:
    def __init__(self, q_F, d_F, q_dense, d_dense, dec_w, target_idx):
        self.q_F, self.d_F, self.q_dense, self.d_dense, self.dec_w = q_F, d_F, q_dense, d_dense, dec_w
        self.max_steps = 3
        self.num_q, self.num_d = len(q_F), len(d_F)
        
        # We STILL need target masks ONLY for calculating the reward during offline training!
        # The agent never SEES this target mask in its state.
        self.target_mask = torch.zeros((self.num_q, self.num_d), dtype=torch.bool, device=DEVICE)
        for i, t in enumerate(target_idx): self.target_mask[i, t] = True

    def _state(self):
        # 1. Active Facet Selection
        facet_vals = self.expansion_candidates.gather(1, self.active_idx)
        
        # 2. Retrieval Entropy (Observable Heuristic)
        scores = torch.matmul(self.q_curr, self.d_dense.T)
        top_k_scores = torch.topk(scores, 10, dim=1).values
        probs = F_func.softmax(top_k_scores, dim=1)
        entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=1).unsqueeze(1)
        
        # 3. Facet Variance
        variance = facet_vals.var(dim=1, unbiased=False).unsqueeze(1)
        
        # NO RANK OR DELTA. ONLY OBSERVABLES.
        return torch.cat([self.q_curr.detach(), facet_vals, entropy, variance], dim=1)

    def reset(self):
        self.step_cnt = 0
        self.M = torch.zeros_like(self.q_F)
        self.q_curr = self.q_dense.clone()
        
        scores = torch.matmul(self.q_curr, self.d_dense.T)
        top_docs = self.d_F[torch.topk(scores, 10, dim=1).indices]
        w = F_func.softmax(torch.arange(10, 0, -1, dtype=torch.float, device=DEVICE), dim=0).view(1, 10, 1)
        self.expansion_candidates = self.q_F + (top_docs * w).sum(dim=1)
        self.active_idx = torch.topk(self.expansion_candidates, 128, dim=1).indices
        
        self.best_targets = scores.masked_fill(~self.target_mask, -1e4).max(dim=1).values
        self.curr_ranks = (scores > self.best_targets.unsqueeze(1)).sum(dim=1) + 1
        
        return self._state()

    def step(self, actions):
        self.step_cnt += 1
        self.M = 0.5 * self.M
        self.M.scatter_add_(1, self.active_idx, 0.2 * actions)
        self.M = torch.clamp(self.M, -2.0, 2.0)
        
        dense_shift = torch.matmul(self.M, self.dec_w.T)
        self.q_curr = F_func.normalize(self.q_dense + (0.3 * dense_shift), p=2, dim=1)
        
        # Reward Calculation (IPS Click Model)
        new_scores = torch.matmul(self.q_curr, self.d_dense.T)
        new_best = new_scores.masked_fill(~self.target_mask, -1e4).max(dim=1).values
        new_ranks = (new_scores > new_best.unsqueeze(1)).sum(dim=1) + 1
        
        # IPS Click Probability = 1 / log2(rank + 1)
        old_ips = 1.0 / torch.log2(self.curr_ranks.float() + 1.0)
        new_ips = 1.0 / torch.log2(new_ranks.float() + 1.0)
        reward = (new_ips - old_ips) * 100.0  # Scale up for PPO
        
        self.curr_ranks = new_ranks
        dones = torch.full((self.num_q,), self.step_cnt >= self.max_steps, dtype=torch.bool, device=DEVICE)
        
        return self._state(), reward, dones, new_ranks

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim=128, hidden_dim=256):
        super().__init__()
        self.actor = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, action_dim), nn.Tanh())
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))
    
    def evaluate(self, states, actions):
        mean = self.actor(states)
        std = torch.exp(torch.clamp(self.log_std, -2.0, 0.5)).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        return dist.log_prob(actions).sum(dim=-1), self.critic(states).squeeze(-1), dist.entropy().sum(dim=-1)
    
    def act(self, states):
        mean = self.actor(states)
        std = torch.exp(torch.clamp(self.log_std, -2.0, 0.5)).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        act = dist.sample()
        return act.detach(), dist.log_prob(act).sum(dim=-1).detach()

ppo_policy = ActorCritic(state_dim=898, action_dim=128).to(DEVICE)
opt = optim.Adam(ppo_policy.parameters(), lr=1e-3)
env = LeakFreeSteeringEnv(ms_q_F, ms_d_F, ms_q_dense, ms_d_dense, dec_weight, ms_targets)

print("\nTraining PPO Agent offline on MS MARCO...")
for ep in tqdm(range(200), desc="Offline PPO Episodes"):
    mem = {'s':[], 'a':[], 'logp':[], 'r':[], 'd':[]}
    s = env.reset()
    for _ in range(env.max_steps):
        a, logp = ppo_policy.act(s)
        ns, r, d, _ = env.step(a)
        mem['s'].append(s); mem['a'].append(a); mem['logp'].append(logp); mem['r'].append(r); mem['d'].append(d)
        s = ns
    
    # Simple PPO Update
    old_s = torch.cat(mem['s']); old_a = torch.cat(mem['a']); old_logp = torch.cat(mem['logp'])
    ret = torch.zeros_like(r)
    discounted = []
    for step_r, step_d in zip(reversed(mem['r']), reversed(mem['d'])):
        ret = step_r + 0.99 * ret * (~step_d).float()
        discounted.insert(0, ret)
    discounted = torch.cat(discounted)
    discounted = (discounted - discounted.mean()) / (discounted.std() + 1e-8)
    
    for _ in range(4):
        logp, val, ent = ppo_policy.evaluate(old_s, old_a)
        adv = discounted - val.detach()
        ratio = torch.exp(logp - old_logp)
        surr1 = ratio * adv
        surr2 = torch.clamp(ratio, 0.8, 1.2) * adv
        loss = -torch.min(surr1, surr2) + 0.5 * nn.MSELoss()(val, discounted) - 0.01 * ent
        opt.zero_grad(); loss.mean().backward(); opt.step()


In [ ]:
# ==============================================================================
# STEP 5: TRUE ZERO-SHOT EVALUATION ON ARGUANA
# ==============================================================================
print("\nEvaluating Zero-Shot on ArguAna...")

with torch.no_grad():
    arg_q_F = F_func.normalize(sae_layer(arg_q_dense)[0], p=2, dim=1)
    arg_d_F = F_func.normalize(sae_layer(arg_d_dense)[0], p=2, dim=1)

eval_env = LeakFreeSteeringEnv(arg_q_F, arg_d_F, arg_q_dense, arg_d_dense, dec_weight, arg_targets)

with torch.no_grad():
    ppo_policy.eval()
    s = eval_env.reset()
    needs_rescue = (eval_env.curr_ranks > 10)
    
    for _ in range(eval_env.max_steps):
        a = ppo_policy.actor(s) # deterministic
        s, _, _, final_ranks = eval_env.step(a)
        
    rescued = needs_rescue & (final_ranks <= 10)
    rescue_rate = rescued.sum().float() / needs_rescue.sum().float() * 100

final_scores = torch.matmul(eval_env.q_curr, arg_d_dense.T)
rl_ndcgs = [compute_ndcg_at_k(torch.topk(final_scores[i], 10).indices.tolist(), arg_targets[i], 10) for i in range(len(arg_q))]

print("\n" + "="*80)
print(" FINAL TRUE ZERO-SHOT RESULTS (ARGUANA)")
print("="*80)
print(f"Base Contriever NDCG@10:  {np.mean(base_ndcgs):.4f}")
print(f"Rocchio Vector PRF:       {rocchio_ndcg:.4f}")
print(f"HyDE (flan-t5-small):     {hyde_ndcg:.4f}")
print(f"RL-Steered (Ours):        {np.mean(rl_ndcgs):.4f}  <-- Zero-Shot from MS MARCO")
print(f"Rescue Rate (Failed->Top10): {rescue_rate:.1f}%")
print("="*80)
